# Portfolio ETL Notebook
## Notebook Goal
This notebook follows a structured ETL workflow for cleaning and preparing the portfolio ETL demo dataset using reusable logic that can later be modularized into extract, transform, and load scripts.

## 1. Project Setup

This section initializes the notebook environment, imports required libraries, and loads reusable ETL modules from the project structure. It also defines dataset-specific variables, resolves repository paths dynamically, and ensures output folders exist before processing begins.

Using dynamic path handling keeps the notebook portable across environments while maintaining a consistent folder structure for raw data, processed outputs, records, logs, and rejected data.

In [ ]:
# --- Standard Library Imports ---
import logging
import re
import sys
from pathlib import Path

# --- Third-Party Imports ---
import chardet
import matplotlib.pyplot as plt
import pandas as pd

# --- Bootstrap: add ETLDevProjects/RH to sys.path for module imports ---
# Uses a minimal inline walk since find_repo_root hasn't been imported yet
def _find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError("Repo root not found.")

_rh_root = _find_repo_root(Path.cwd()) / "ETLDevProjects" / "RH"
if str(_rh_root) not in sys.path:
    sys.path.insert(0, str(_rh_root))

# --- Module Imports ---
from utils.path_utils      import find_repo_root
from extract.read_settings import check_file_exists, detect_encoding, load_raw_data
from transform.validate    import clean_placeholder, is_valid_name
from load.export_utils     import export_cleaned, summarise_garbage, log_etl_complete

# --- Project Variables ---
project_name = "portfolio_etl_demo"
dataset_name = "portfolio_etl_demo"
dataset_file = "portfolio_etl_demo.csv"

# --- Repo Root ---
repo_root = find_repo_root(Path.cwd())

# --- Paths ---
raw_file      = repo_root / "datasets" / project_name / "raw" / dataset_file
records_dir   = repo_root / "datasets" / project_name / "records"
processed_dir = repo_root / "datasets" / project_name / "processed"
garbage_dir   = repo_root / "datasets" / project_name / "garbage"
outputs_dir   = repo_root / "datasets" / project_name / "outputs"
logs_dir      = repo_root / "datasets" / project_name / "logs"

# --- Create Folders if Needed ---
for folder in [records_dir, processed_dir, garbage_dir, outputs_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project   : {project_name}")
print(f"Repo root : {repo_root}")
print(f"Raw file  : {raw_file}")
print(f"Folders ready: records, processed, garbage, outputs, logs")

## 1.1 Dataset Configuration
All dataset-specific settings are defined here and should be reviewed before running the pipeline.

This includes file naming, delimiter settings, retained columns, placeholder values, validation patterns, and completeness scoring rules.

Some downstream cells still reference specific field names directly. If you change `columns_to_keep`, update the corresponding field references in later cleaning, validation, and visualization sections.

This project is structured so that reusable controls are centralized here, while dataset-specific transformation logic remains explicit in later sections.

In [ ]:
# =============================================================
# Dataset Configuration — adjust these before running
# =============================================================
# Downstream cells reference these column names directly.
# If you change columns_to_keep, update the field references
# in sections 7–10 and any validation/visualization logic.

# --- Delimiter ---
delimiter = ";"  # set to the field separator used in your source file

# --- Column Scope ---
# id             — internal database key, no analytical value
# algorithm      — password hashing method, not needed in cleaned output
# is_active      — account state flag, out of scope
# is_super_admin — privilege flag, out of scope
# last_login     — 81.86% empty, out of scope
# created_at     — out of scope
columns_to_drop = ["id", "algorithm", "is_active", "is_super_admin", "last_login", "created_at"]
columns_to_keep = ["first_name", "last_name", "email_address", "salt", "password"]

# --- Placeholder Values ---
PLACEHOLDERS = {"null", "none", "n/a", "na", "unknown", "undefined", "-", ".", " "}

# --- Validation Rules ---
EMAIL_REGEX = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
NAME_REGEX  = r"^[^\W\d_]+(?:[ '\-][^\W\d_]+)*$"
NAME_MIN    = 1
NAME_MAX    = 70

# --- Deduplication Weights ---
# Keys must match columns_to_keep. Higher weight = stronger preference
# for rows where that field is present when resolving email duplicates.
COMPLETENESS_WEIGHTS = {
    "first_name"    : 2,
    "last_name"     : 2,
    "email_address" : 3,
    "salt"          : 1,
    "password"      : 2,
}

# --- Visualization Fields ---
# Columns shown in the Field Completeness chart (Section 11.1)
viz_fields = ["first_name", "last_name", "email_address", "salt", "password"]

## 2. Logger Setup
Logging is configured to capture both detailed file-based diagnostics and concise console output during execution. Separate handlers are used so that full debug activity is written to a log file while key progress messages remain visible in the notebook.

A handler check is included to prevent duplicate log entries when cells are re-run during development.

In [ ]:
# --- Log File Path ---
log_file = logs_dir / f"{dataset_name}_cleaning.log"

# --- Logger Configuration ---
logger = logging.getLogger(dataset_name)
logger.setLevel(logging.DEBUG)

# Clear any existing handlers before adding fresh ones
if logger.hasHandlers():
    logger.handlers.clear()

# File handler — full debug log
fh = logging.FileHandler(log_file, encoding="utf-8")
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))

# Console handler — info and above
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))

logger.addHandler(fh)
logger.addHandler(ch)

logger.info(f"Logger initialised → {log_file}")
logger.info(f"Starting ETL for dataset: {dataset_name}")

## 3. File Discovery / Read Settings
Before loading the dataset, the source file is verified, encoding is detected, and read settings are confirmed. Because the delimiter is already known for this source, it is assigned directly rather than inferred.

This step reduces read-time errors and ensures the file is interpreted consistently before transformation begins.

In [ ]:
# --- File Existence Check ---
check_file_exists(raw_file, logger)

# --- Encoding Detection ---
file_encoding = detect_encoding(raw_file, logger=logger)

# --- Delimiter ---
logger.info(f"Delimiter set: '{delimiter}' (configured in Dataset Configuration)")

# --- Read Settings Summary ---
print(f"File     : {raw_file.name}")
print(f"Encoding : {file_encoding}")
print(f"Delimiter: '{delimiter}'")

## 4. Load Raw Data
The raw source file is loaded into a dataframe using the validated read settings established earlier. An initial preview is displayed to confirm that the structure and contents align with expectations before inspection begins.

In [ ]:
# --- Load Raw Data ---
df_raw = load_raw_data(raw_file, delimiter, file_encoding, logger)

# --- First Preview ---
df_raw.head(3)

## 5. Raw Data Inspection
A baseline inspection is performed to understand dataset size, field types, empty-value distribution, and duplicate patterns before any transformations are applied.

This provides an initial quality snapshot and helps identify structural issues that may affect downstream cleaning, validation, and deduplication decisions.

In [ ]:
# --- Shape ---
print("=== Shape ===")
print(f"{df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns\n")

# --- Column Names & Dtypes ---
print("=== Columns & Dtypes ===")
print(df_raw.dtypes.to_string(), "\n")

# --- Missing Values ---
print("=== Missing Values ===")
missing = (df_raw == "").sum()  # Source loaded as string values, so empty fields appear as "" rather than NaN
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_report = pd.DataFrame({"empty_count": missing, "empty_%": missing_pct})
print(missing_report.to_string(), "\n")

# --- Duplicate Snapshot ---
print("=== Duplicates ===")
full_dupes = df_raw.duplicated().sum()
email_dupes = df_raw["email_address"].duplicated().sum()
print(f"Full row duplicates : {full_dupes:,}")
print(f"Duplicate emails    : {email_dupes:,}")

logger.info(f"Inspection complete — full dupes: {full_dupes:,} | duplicate emails: {email_dupes:,}")

## 6. Column Selection / Scope Decision
Only fields required for the cleaning objective are retained at this stage. Columns outside the current scope, including internal identifiers, system flags, and low-value administrative fields, are excluded to keep the working dataset focused and easier to validate.

A checkpoint file is saved after selection to preserve the reduced working version before transformation begins.

In [ ]:
# --- Column lists defined in Dataset Configuration ---
df = df_raw[columns_to_keep].copy()

logger.info(f"Columns dropped: {columns_to_drop}")
logger.info(f"Columns kept: {columns_to_keep}")

print(f"Columns kept  : {list(df.columns)}")
print(f"Shape after   : {df.shape[0]:,} rows × {df.shape[1]} columns")

# --- Checkpoint: records ---
checkpoint = records_dir / f"{dataset_name}_post_column_select.csv"
df.to_csv(checkpoint, index=False)
logger.info(f"Checkpoint saved → {checkpoint}")
print(f"Checkpoint saved → {checkpoint.name}")

df.head()

## 7. Standardize Structure
Structural consistency is applied before field-level cleaning by confirming column naming conventions, removing leading and trailing whitespace, normalizing internal spacing in name fields, and standardizing email casing.

A checkpoint is saved after standardization so that structural adjustments are preserved before content-level cleaning begins.

In [ ]:
# --- Column Names ---
# Already snake_case — no changes needed
df.columns = df.columns.str.strip().str.lower()
logger.info(f"Column names confirmed: {list(df.columns)}")

# --- Strip Whitespace (all columns) ---
df = df.apply(lambda col: col.str.strip())

# --- Normalize Internal Whitespace (name fields only) ---
for col in ["first_name", "last_name"]:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

# --- Lowercase Email ---
df["email_address"] = df["email_address"].str.lower()

logger.info("Structure standardised: whitespace stripped, names normalised, emails lowercased")

# --- Confirm ---
print(f"Columns : {list(df.columns)}")
print(f"Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")

# --- Checkpoint: records ---
checkpoint = records_dir / f"{dataset_name}_post_standardize.csv"
df.to_csv(checkpoint, index=False)
logger.info(f"Checkpoint saved → {checkpoint}")
print(f"Checkpoint saved → {checkpoint.name}")

df.head()

## 8. Clean Values
Field-level cleaning is applied to standardize placeholder values, remove invalid characters, and normalize text formatting across key columns.

Name fields are cleaned to preserve only expected alphabetic characters and punctuation, email values are stripped of internal spacing, and placeholder values are converted consistently before validation.

In [ ]:
# -------------------------------------------------------
# first_name
# -------------------------------------------------------
df["first_name"] = df["first_name"].apply(lambda v: clean_placeholder(v, PLACEHOLDERS))
df["first_name"] = df["first_name"].str.replace(r"[^a-zA-Z\s'\-]", "", regex=True)
df["first_name"] = df["first_name"].str.strip()
df["first_name"] = df["first_name"].str.title()

logger.info(f"first_name cleaned — empty after cleaning: {(df['first_name'] == '').sum():,}")

# -------------------------------------------------------
# last_name
# -------------------------------------------------------
df["last_name"] = df["last_name"].apply(lambda v: clean_placeholder(v, PLACEHOLDERS))
df["last_name"] = df["last_name"].str.replace(r"[^a-zA-Z\s'\-]", "", regex=True)
df["last_name"] = df["last_name"].str.strip()
df["last_name"] = df["last_name"].str.title()

logger.info(f"last_name cleaned — empty after cleaning: {(df['last_name'] == '').sum():,}")

# -------------------------------------------------------
# email_address
# -------------------------------------------------------
df["email_address"] = df["email_address"].apply(lambda v: clean_placeholder(v, PLACEHOLDERS))
df["email_address"] = df["email_address"].str.replace(r"\s+", "", regex=True)

logger.info(f"email_address cleaned — empty after cleaning: {(df['email_address'] == '').sum():,}")

# -------------------------------------------------------
# salt & password — dataset-specific: hex hashes, values must not be modified
# -------------------------------------------------------
df["salt"]     = df["salt"].apply(lambda v: clean_placeholder(v, PLACEHOLDERS))
df["password"] = df["password"].apply(lambda v: clean_placeholder(v, PLACEHOLDERS))

logger.info(f"salt — empty after cleaning: {(df['salt'] == '').sum():,}")
logger.info(f"password — empty after cleaning: {(df['password'] == '').sum():,}")

print("=== Post-Clean Empty Counts (before name split) ===")
print((df == "").sum().to_string())

## 8.1 Name Reconstruction (Dataset-Specific Rule)
Some records contain full names within the `first_name` field while `last_name` is empty. For these cases, a Spanish naming rule is applied: the final two words are assigned to `last_name`, while preceding words remain in `first_name`.

Records with five or more name parts are flagged for review because name ambiguity increases beyond standard surname patterns.

In [ ]:
# --- Name Split: first_name contains full name, last_name is empty ---
# Applies only when last_name is empty and first_name contains multiple words.
# Spanish naming convention: [First name(s)] [Paternal apellido] [Maternal apellido]
# Last 2 words → last_name, everything before → first_name
# 5+ word names are flagged for review due to increased ambiguity

df["name_split_flagged"] = False

mask = (df["last_name"] == "") & (df["first_name"].str.contains(r"\s", regex=True))
split_count = mask.sum()

def split_spanish_name(name):
    parts = name.split()
    if len(parts) >= 3:
        first = " ".join(parts[:-2])
        last  = " ".join(parts[-2:])
    elif len(parts) == 2:
        first = parts[0]
        last  = parts[1]
    else:
        first = parts[0]
        last  = ""
    flagged = len(parts) >= 5
    return first, last, flagged

def apply_split(row):
    first, last, flagged = split_spanish_name(row["first_name"])
    row["first_name"]        = first
    row["last_name"]         = last
    row["name_split_flagged"] = flagged
    return row

df.loc[mask] = df.loc[mask].apply(apply_split, axis=1)

flagged_count = df["name_split_flagged"].sum()

logger.info(f"Name split applied to {split_count:,} rows")
logger.info(f"Rows flagged (5+ word names): {flagged_count:,}")

print(f"Rows split          : {split_count:,}")
print(f"Rows flagged        : {flagged_count:,}")

print("\n=== Post-Split Empty Counts ===")
print((df[["first_name", "last_name", "email_address", "salt", "password"]] == "").sum().to_string())

# --- Checkpoint: records ---
checkpoint = records_dir / f"{dataset_name}_post_clean.csv"
df.to_csv(checkpoint, index=False)
logger.info(f"Checkpoint saved → {checkpoint}")
print(f"\nCheckpoint saved → {checkpoint.name}")

## 9. Validation
Dataset records are validated against dataset-specific rules:

1. **Email format** — rows with invalid email syntax are rejected.  
2. **Missing password** — rows without a password hash are rejected.  
3. **Name fields** — non-empty names are checked against regex and length rules; invalid names are cleared rather than immediately discarded (soft invalidation).  
4. **Insufficient PII** — rows with fewer than two of `first_name`, `last_name`, or `email_address` are rejected.

Rejected rows are saved to the `garbage` folder for traceability, and a validated milestone dataset is produced for downstream processing.

In [ ]:
df_remaining = df.copy()

# --- Re-normalise whitespace in name fields post-split ---
# The name split in Section 8 may introduce double spaces not caught by Section 7
for col in ["first_name", "last_name"]:
    df_remaining[col] = df_remaining[col].str.replace(r"\s+", " ", regex=True).str.strip()

# -------------------------------------------------------
# 1. Invalid email format
# -------------------------------------------------------
mask_invalid_email = ~df_remaining["email_address"].str.match(EMAIL_REGEX, na=False)
df_invalid_email   = df_remaining[mask_invalid_email].copy()
df_remaining       = df_remaining[~mask_invalid_email].copy()

path = garbage_dir / f"{dataset_name}_invalid_email.csv"
df_invalid_email.to_csv(path, index=False)
logger.info(f"Invalid email → {len(df_invalid_email):,} rows → {path.name}")
print(f"Invalid email       : {len(df_invalid_email):,} rows")

# -------------------------------------------------------
# 2. Missing password — dataset-specific: schema requires a password hash
# -------------------------------------------------------
mask_missing_pw = df_remaining["password"] == ""
df_missing_pw   = df_remaining[mask_missing_pw].copy()
df_remaining    = df_remaining[~mask_missing_pw].copy()

path = garbage_dir / f"{dataset_name}_missing_password.csv"
df_missing_pw.to_csv(path, index=False)
logger.info(f"Missing password → {len(df_missing_pw):,} rows → {path.name}")
print(f"Missing password    : {len(df_missing_pw):,} rows")

# -------------------------------------------------------
# 3. Name validation — soft invalidation
# Checks non-empty name fields against regex and length rules.
# Invalid fields are emptied rather than immediately rejecting the row.
# Rows that then fall below 2 PII fields are caught in step 4.
# -------------------------------------------------------
for col in ["first_name", "last_name"]:
    invalid_mask = ~df_remaining[col].apply(
        lambda v: is_valid_name(v, NAME_REGEX, NAME_MIN, NAME_MAX)
    )
    df_remaining.loc[invalid_mask, col] = ""

invalid_name_mask = ~df.loc[df_remaining.index, ["first_name", "last_name"]].apply(
    lambda row: (
        is_valid_name(row["first_name"], NAME_REGEX, NAME_MIN, NAME_MAX)
        and is_valid_name(row["last_name"], NAME_REGEX, NAME_MIN, NAME_MAX)
    ),
    axis=1,
)
name_fields_invalidated = invalid_name_mask.sum()

path = garbage_dir / f"{dataset_name}_invalid_name.csv"
df.loc[df_remaining.index[invalid_name_mask]].to_csv(path, index=False)
logger.info(f"Name fields soft-invalidated: {name_fields_invalidated:,} rows → {path.name}")
print(f"Name fields cleared : {name_fields_invalidated:,} rows")

# -------------------------------------------------------
# 4. Insufficient PII — dataset-specific: threshold and columns defined for this schema
# -------------------------------------------------------
pii_cols     = ["first_name", "last_name", "email_address"]
pii_score    = df_remaining[pii_cols].apply(lambda col: col != "").sum(axis=1)
mask_low_pii = pii_score < 2
df_low_pii   = df_remaining[mask_low_pii].copy()
df_remaining = df_remaining[~mask_low_pii].copy()

path = garbage_dir / f"{dataset_name}_insufficient_pii.csv"
df_low_pii.to_csv(path, index=False)
logger.info(f"Insufficient PII → {len(df_low_pii):,} rows → {path.name}")
print(f"Insufficient PII    : {len(df_low_pii):,} rows")

# -------------------------------------------------------
# Summary
# -------------------------------------------------------
total_rejected = len(df_invalid_email) + len(df_missing_pw) + len(df_low_pii)
df_valid = df_remaining.copy()

logger.info(f"Validation complete — valid: {len(df_valid):,} | rejected: {total_rejected:,}")
print(f"\nTotal rejected      : {total_rejected:,}")
print(f"Valid rows          : {len(df_valid):,}")

# --- Milestone: processed ---
milestone = processed_dir / f"{dataset_name}_validated.csv"
df_valid.to_csv(milestone, index=False)
logger.info(f"Milestone saved → {milestone}")
print(f"Milestone saved → {milestone.name}")

## 10. Deduplication
Duplicate records are removed in two steps:

1. **Full-row duplicates** — identical rows across all columns are removed, keeping the first occurrence.  
2. **Email-based duplicates** — when multiple rows share the same `email_address`, the row with the highest weighted completeness score (based on presence of key fields) is retained, preserving original order for ties.  

Rows removed during email deduplication are saved for traceability, and a deduplicated milestone dataset is produced.

In [ ]:
# -------------------------------------------------------
# 1. Full Row Deduplication
# -------------------------------------------------------
full_dupe_mask  = df_valid.duplicated(keep="first")
full_dupe_count = full_dupe_mask.sum()
df_valid        = df_valid[~full_dupe_mask].copy()

logger.info(f"Full row duplicates removed: {full_dupe_count:,}")
print(f"Full row duplicates removed : {full_dupe_count:,}")

# -------------------------------------------------------
# 2. Weighted Completeness Score
# -------------------------------------------------------
df_valid["_completeness"] = sum(
    (df_valid[col] != "").astype(int) * weight
    for col, weight in COMPLETENESS_WEIGHTS.items()
)

# -------------------------------------------------------
# 3. Sort by Score — stable sort preserves original order on ties
# -------------------------------------------------------
df_sorted = df_valid.sort_values("_completeness", ascending=False, kind="stable")

# -------------------------------------------------------
# 4. Email Deduplication — keep most complete row per email
# -------------------------------------------------------
email_dupe_count = df_sorted.duplicated(subset=["email_address"], keep=False).sum()
df_dupes         = df_sorted[df_sorted.duplicated(subset=["email_address"], keep="first")].copy()
df_clean         = df_sorted[~df_sorted.duplicated(subset=["email_address"], keep="first")].copy()

# Drop working column
df_clean = df_clean.drop(columns=["_completeness"])
df_dupes = df_dupes.drop(columns=["_completeness"])

# Restore original order
df_clean = df_clean.sort_index()

path = garbage_dir / f"{dataset_name}_duplicate_email.csv"
df_dupes.to_csv(path, index=False)
logger.info(f"Email duplicates → {len(df_dupes):,} rows removed → {path.name}")
print(f"Email duplicates involved   : {email_dupe_count:,}")
print(f"Duplicate rows removed      : {len(df_dupes):,}")

# -------------------------------------------------------
# Summary
# -------------------------------------------------------
logger.info(f"Deduplication complete — rows before: {len(df_valid):,} | rows after: {len(df_clean):,}")
print(f"\nRows before dedup   : {len(df_valid):,}")
print(f"Rows after dedup    : {len(df_clean):,}")

# --- Milestone: processed ---
milestone = processed_dir / f"{dataset_name}_deduped.csv"
df_clean.to_csv(milestone, index=False)
logger.info(f"Milestone saved → {milestone}")
print(f"Milestone saved → {milestone.name}")

## 11. Final Quality Review
A final inspection of the cleaned dataset is performed to confirm structure, missing values, uniqueness, and flagged records.  

Key metrics such as raw vs rejected rows, rejection reasons, and overall retention rate are summarized to provide a concise overview of data quality before export. This step ensures confidence in the final dataset and transparency in ETL decisions.

In [ ]:
# --- Shape ---
print("=== Final Shape ===")
print(f"{df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns\n")

# --- Missing Values ---
print("=== Missing Values ===")
missing     = (df_clean == "").sum()
missing_pct = (missing / len(df_clean) * 100).round(2)
print(pd.DataFrame({"empty_count": missing, "empty_%": missing_pct}).to_string(), "\n")

# --- Unique Value Percentages ---
print("=== Unique Values ===")
for col in df_clean.columns:
    n_unique = df_clean[col].nunique()
    pct      = round(n_unique / len(df_clean) * 100, 2)
    print(f"  {col:<25} {n_unique:>10,} unique  ({pct}%)")

# --- Value Counts: name_split_flagged ---
print("\n=== Name Split Flagged ===")
print(df_clean["name_split_flagged"].value_counts().to_string())

# --- ETL Summary ---
print("\n=== ETL Summary ===")
total_rejected = df_raw.shape[0] - len(df_clean)
print(f"  Raw rows              : {df_raw.shape[0]:>12,}")
print(f"  Invalid email         : {len(df_invalid_email):>12,}")
print(f"  Missing password      : {len(df_missing_pw):>12,}")
print(f"  Name fields cleared   : {name_fields_invalidated:>12,}")
print(f"  Insufficient PII      : {len(df_low_pii):>12,}")
print(f"  Duplicate email       : {len(df_dupes):>12,}")
print(f"  ─────────────────────────────────")
print(f"  Total rejected        : {total_rejected:>12,}")
print(f"  Final clean rows      : {len(df_clean):>12,}")
print(f"  Retention rate        : {round(len(df_clean) / df_raw.shape[0] * 100, 2):>11}%")

logger.info(f"Final quality review complete — {len(df_clean):,} rows ready for export")

### 11.1 ETL Quality Visualizations
Visualizations provide a clear snapshot of the ETL process and dataset quality:

1. **ETL Funnel** — shows row counts at key pipeline stages: raw, post-validation, and post-deduplication.  
2. **Field Completeness** — highlights the proportion of non-empty vs empty values across key columns.  
3. **Deduplication Breakdown** — illustrates how duplicates were handled, including kept and removed rows.  

These visuals complement the tabular summary, making it easier to quickly assess data quality and processing outcomes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Portfolio ETL Demo — ETL Quality Report", fontsize=14, fontweight="bold", y=1.02)

# -------------------------------------------------------
# 1. ETL Funnel
# -------------------------------------------------------
funnel_labels = ["Raw", "Post-Validation", "Post-Dedup\n(Final Clean)"]
funnel_values = [
    df_raw.shape[0],
    len(df_valid),
    len(df_clean)
]

colors_funnel = ["#5b9bd5", "#70ad47", "#2e7d32"]
bars = axes[0].barh(funnel_labels, funnel_values, color=colors_funnel, edgecolor="white", height=0.5)
axes[0].set_title("ETL Funnel", fontweight="bold")
axes[0].set_xlabel("Row Count")
axes[0].invert_yaxis()
for bar, val in zip(bars, funnel_values):
    axes[0].text(bar.get_width() * 0.98, bar.get_y() + bar.get_height() / 2,
                 f"{val:,}", va="center", ha="right", color="white", fontsize=9, fontweight="bold")

# -------------------------------------------------------
# 2. Field Completeness
# -------------------------------------------------------
fields       = viz_fields
valid_counts = [(df_clean[col] != "").sum() for col in fields]
empty_counts = [(df_clean[col] == "").sum() for col in fields]
total        = len(df_clean)

valid_pct = [v / total * 100 for v in valid_counts]
empty_pct = [e / total * 100 for e in empty_counts]

axes[1].barh(fields, valid_pct, color="#70ad47", label="Valid", edgecolor="white")
axes[1].barh(fields, empty_pct, left=valid_pct, color="#e05c5c", label="Empty", edgecolor="white")
axes[1].set_title("Field Completeness", fontweight="bold")
axes[1].set_xlabel("% of Rows")
axes[1].set_xlim(0, 100)
axes[1].legend(loc="lower right", fontsize=8)
axes[1].invert_yaxis()
for i, (v, e) in enumerate(zip(valid_pct, empty_pct)):
    if v > 5:
        axes[1].text(v / 2, i, f"{v:.1f}%", va="center", ha="center", color="white", fontsize=8)
    if e > 5:
        axes[1].text(v + e / 2, i, f"{e:.1f}%", va="center", ha="center", color="white", fontsize=8)

# -------------------------------------------------------
# 3. Deduplication Breakdown
# -------------------------------------------------------
dedup_labels = ["Unique Rows\n(No Duplicates)", "Kept\n(Best Version)", "Removed\n(Duplicates)"]
dedup_values = [len(df_clean) - (email_dupe_count // 2), len(df_dupes), len(df_dupes)]

total_emails_duped = email_dupe_count
unique_no_dupe     = len(df_clean) - (email_dupe_count - len(df_dupes))
kept_best          = email_dupe_count - len(df_dupes)
removed            = len(df_dupes)

dedup_labels = ["Unique\n(No Duplicate)", "Kept\n(Best Version)", "Removed"]
dedup_values = [unique_no_dupe, kept_best, removed]
colors_dedup = ["#70ad47", "#5b9bd5", "#e05c5c"]

bars = axes[2].bar(dedup_labels, dedup_values, color=colors_dedup, edgecolor="white", width=0.5)
axes[2].set_title("Deduplication Breakdown", fontweight="bold")
axes[2].set_ylabel("Row Count")
for bar, val in zip(bars, dedup_values):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1000,
                 f"{val:,}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(outputs_dir / f"{dataset_name}_quality_report.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("Quality report visualisation saved to outputs/")

## 12. Export
The cleaned dataset is exported to the `outputs` folder, while all rejected or intermediary rows remain in the `garbage` and `records` folders for traceability.

A summary of exported files, checkpoints, and garbage rows is printed for quick reference, and a final log entry marks ETL completion. This ensures transparency and reproducibility for both portfolio demonstration and production use.

In [ ]:
# --- Cleaned Output ---
output_file = export_cleaned(df_clean, outputs_dir, dataset_name, logger)
print(f"Cleaned output      → {output_file.name}")

# --- Garbage Files Summary ---
print("\n=== Garbage Files ===")
garbage_summary = summarise_garbage(garbage_dir, dataset_name, logger)
for name, count in garbage_summary:
    print(f"  {name:<45} {count:>10,} rows")

# --- Records & Processed Summary ---
print("\n=== Checkpoints ===")
for folder, label in [(records_dir, "records"), (processed_dir, "processed")]:
    for f in sorted(folder.glob(f"{dataset_name}_*.csv")):
        print(f"  [{label}] {f.name}")

# --- Final Log Summary ---
log_etl_complete(
    dataset_name   = dataset_name,
    raw_row_count  = df_raw.shape[0],
    clean_row_count= len(df_clean),
    output_file    = output_file,
    garbage_count  = len(garbage_summary),
    logger         = logger,
)

print(f"\n{'=' * 50}")
print(f"  ETL complete — {len(df_clean):,} rows exported to {output_file.name}")
print(f"{'=' * 50}")